# Task 3: Research Results, Data Analysis, and Scientific Visualization
**Project:** Fast Dopamine & Academic Performance — GPA Prediction Benchmark  
**Repository:** `https://github.com/EraMiir/fast-dopamine-academic-performance`

This notebook executes the complete reproducible analysis pipeline by importing the modular functions from `src/`:
1. **Data Ingestion & Warm-up Filtering (`src/process_data.py`):** Loading raw benchmark logs (`data/raw/test_run.jsonl`) generated from the real student survey dataset (`data/cleaned_survey.csv`, $N=256$, 24 features) and discarding cold-start `warmup` runs.
2. **Statistical Aggregation (`src/process_data.py`):** Computing Mean, SD, Median, Percentiles ($p_{25}, p_{75}, p_{95}, p_{99}$), 1,000-iteration Bootstrapped 95% Confidence Intervals (CI), and Guardrail Error Rates.
3. **Scientific Visualization & Export (`src/visualize.py`):** Generating colorblind-safe plots with explicit physical units, log scales, and 95% CI bands exported to `.pdf` and 300 DPI `.png` in `reports/figures/`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Fix random seed for 100% reproducibility
np.random.seed(42)

# Add src/ to Python path and import modular pipeline functions
root_dir = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
sys.path.insert(0, str(root_dir / 'src'))

from process_data import process_benchmark_logs
from visualize import generate_all_plots

# 1. Data Ingestion & Warm-up Filtering
raw_jsonl_path = str(root_dir / 'data/raw/test_run.jsonl')
processed_csv_path = str(root_dir / 'data/processed/metrics_summary.csv')

raw_df, eval_df, summary_df = process_benchmark_logs(
    raw_jsonl_path=raw_jsonl_path,
    output_csv_path=processed_csv_path,
    n_boot=1000,
    seed=42
)

print(f'Total raw records loaded: {len(raw_df)}')
print(f'Warm-up records discarded: {len(raw_df) - len(eval_df)}')
print(f'Clean evaluation records retained: {len(eval_df)}')

Total raw records loaded: 276
Warm-up records discarded: 36
Clean evaluation records retained: 240


In [2]:
# 2. Inspect Statistical Aggregation Results (including p95, p99, MAE, and 95% Bootstrap CI)
cols_to_show = [
    'model', 'RMSE_mean', 'RMSE_sd', 'RMSE_median', 'RMSE_p95', 'RMSE_p99',
    'MAE_mean', 'latency_median_ms', 'latency_p95_ms', 'latency_p99_ms', 'error_rate'
]
print('Aggregated Metrics Summary (Sample Size N=256):')
print(summary_df[summary_df['sample_size'] == 256][cols_to_show].to_string(index=False))

Aggregated Metrics Summary (Sample Size N=256):
           model  RMSE_mean  RMSE_sd  RMSE_median  RMSE_p95  RMSE_p99  MAE_mean  latency_median_ms  latency_p95_ms  latency_p99_ms  error_rate
  Baseline_Ridge     0.3338   0.0245       0.3335    0.3731    0.3766    0.2677              1.069           1.491           1.601         0.0
GradientBoosting     0.2918   0.0278       0.2919    0.3402    0.3425    0.2192             80.636          94.740          96.930         0.0
    RandomForest     0.2866   0.0220       0.2926    0.3153    0.3160    0.2221            173.454         186.631         187.356         0.0


## 4. Feature–GPA Relationship Analysis

This section examines the relationships between selected student factors and academic performance (GPA). The analysis uses Spearman rank correlation coefficients calculated from the cleaned survey dataset (N = 256). Bootstrap 95% confidence intervals are reported alongside p-values.

The analysis is exploratory. Correlation does not imply causation.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

correlations_path = root_dir / 'data' / 'processed' / 'feature_gpa_correlations.csv'
correlation_df = pd.read_csv(correlations_path)

print('Feature–GPA Correlation Results')
print(f'Number of observations per factor: {correlation_df["n"].iloc[0]}')
display(correlation_df.round(4))


Feature–GPA Correlation Results
Number of observations per factor: 256


,factor,source_column,n,spearman_rho,spearman_ci95_low,spearman_ci95_high,spearman_p_value,pearson_r,direction
0,Study time (hours),Study time (In Hours),256,0.3016,0.1919,0.4087,0.0000,0.2800,Positive
1,Attendance rate,Attendance rate (In Percentile),256,0.2354,0.1187,0.3495,0.0001,0.2818,Positive
2,Academic social media distraction,Social Media Distraction During Academic Activ...,256,-0.1747,-0.2980,-0.0516,0.0051,-0.1536,Negative
3,Sleep disturbance,Sleep Disturbance on Sleep Quality,256,-0.0900,-0.2134,0.0394,0.1511,-0.0901,Weak / near zero
4,Social media time (hours),Time spent in social media (hours),256,-0.0464,-0.1610,0.0687,0.4599,-0.0559,Weak / near zero


In [4]:
# 3. Scientific Visualization & Export (.pdf & 300 DPI .png)
figures_dir = str(root_dir / 'reports/figures')
generate_all_plots(eval_df, summary_df, out_dir=figures_dir)
print(f'Generated fig1_latency_vs_load and fig2_distribution_boxplot (.pdf & .png) in ../reports/figures')

/workspaces/fast-dopamine-academic-performance/src/visualize.py:68: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=eval_df, x="model", y="RMSE", order=order, palette=palette,
/workspaces/fast-dopamine-academic-performance/src/visualize.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax1.set_xticklabels(labels, fontsize=10.5, fontweight="bold")
/workspaces/fast-dopamine-academic-performance/src/visualize.py:85: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=eval_df, x="model", y="latency_ms", order=order, palette=palette,
/workspaces/fast-dopamine-academic-per

Generated fig1_latency_vs_load and fig2_distribution_boxplot (.pdf & .png) in ../reports/figures
